# Formation Matchup Analysis

This notebook reads the CSV outputs generated by `formation_analysis.py` and shows a few quick descriptive checks. Run the script first from the project root:

```bash
python formation_analysis.py
```

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
sys.path.insert(0, str(PROJECT_ROOT))

rows_path = OUTPUT_DIR / 'formation_match_rows.csv'
summary_path = OUTPUT_DIR / 'formation_matchup_summary.csv'
favorable_path = OUTPUT_DIR / 'top_favorable_matchups.csv'
unfavorable_path = OUTPUT_DIR / 'top_unfavorable_matchups.csv'

rows = pd.read_csv(rows_path)
summary = pd.read_csv(summary_path)
favorable = pd.read_csv(favorable_path)
unfavorable = pd.read_csv(unfavorable_path)

rows.head()

c:\Users\Liuhy\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Liuhy\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,match_id,team,opponent,formation,opponent_formation,goals_for,goals_against,goal_diff,result,points,source_file
0,15946,Barcelona,Deportivo Alavés,442,451,3,0,3,W,3,archive\data\events\15946.json
1,15946,Deportivo Alavés,Barcelona,451,442,0,3,-3,L,0,archive\data\events\15946.json
2,15956,Real Valladolid,Barcelona,41212,433,0,1,-1,L,0,archive\data\events\15956.json
3,15956,Barcelona,Real Valladolid,433,41212,1,0,1,W,3,archive\data\events\15956.json
4,15973,Barcelona,Huesca,433,442,8,2,6,W,3,archive\data\events\15973.json


## Most Common Formations

This counts team-match rows, so each match contributes one row per team.

In [2]:
rows['formation'].value_counts().head(15).rename_axis('formation').reset_index(name='team_match_rows')

,formation,team_match_rows
0,4231,2537
1,433,1853
2,442,1296
3,4141,548
4,41212,464
5,352,443
6,343,334
7,4411,304
8,3421,192
9,3412,189


## Formation Matchup Summary

The saved summary applies the minimum sample size filter configured in the script.

In [3]:
summary.sort_values(['points_per_game', 'win_rate', 'goal_diff_per_game'], ascending=False).head(20)

,formation,opponent_formation,games,wins,draws,losses,win_rate,draw_rate,loss_rate,points_per_game,goal_diff_per_game,smoothed_win_rate
83,433,451,22,18,3,1,0.818182,0.136364,0.045455,2.590909,1.772727,0.683884
74,433,343,72,50,6,16,0.694444,0.083333,0.222222,2.166667,1.305556,0.657126
58,4231,343,91,54,25,12,0.593407,0.274725,0.131868,2.054945,1.032967,0.573112
92,442,343,39,25,4,10,0.641026,0.102564,0.256410,2.025641,0.538462,0.589475
78,433,4141,107,63,25,19,0.588785,0.233645,0.177570,2.000000,1.289720,0.571661
6,3412,442,14,8,4,2,0.571429,0.285714,0.142857,2.000000,0.785714,0.495179
75,433,3511,11,6,4,1,0.545455,0.363636,0.090909,2.000000,1.636364,0.470681
9,3421,352,22,14,1,7,0.636364,0.045455,0.318182,1.954545,0.727273,0.558884
8,3421,343,19,11,4,4,0.578947,0.210526,0.210526,1.947368,1.157895,0.513252
57,4231,3421,28,17,3,8,0.607143,0.107143,0.285714,1.928571,0.250000,0.549587


## Top Favorable Matchups For 433

In [4]:
favorable[favorable['formation'].astype(str) == '433'].head(5)

,formation,opponent_formation,games,wins,draws,losses,win_rate,draw_rate,loss_rate,points_per_game,goal_diff_per_game,smoothed_win_rate
40,433,451,22,18,3,1,0.818182,0.136364,0.045455,2.590909,1.772727,0.683884
41,433,343,72,50,6,16,0.694444,0.083333,0.222222,2.166667,1.305556,0.657126
42,433,4141,107,63,25,19,0.588785,0.233645,0.177570,2.000000,1.289720,0.571661
43,433,3511,11,6,4,1,0.545455,0.363636,0.090909,2.000000,1.636364,0.470681
44,433,442,331,183,72,76,0.552870,0.217523,0.229607,1.876133,0.972810,0.548048


## Top Unfavorable Matchups For 433

In [5]:
unfavorable[unfavorable['formation'].astype(str) == '433'].head(5)

,formation,opponent_formation,games,wins,draws,losses,win_rate,draw_rate,loss_rate,points_per_game,goal_diff_per_game,smoothed_win_rate
40,433,433,316,123,70,123,0.389241,0.221519,0.389241,1.389241,0.000000,0.389216
41,433,3421,41,21,3,17,0.512195,0.073171,0.414634,1.609756,0.585366,0.487927
42,433,352,118,58,28,32,0.491525,0.237288,0.271186,1.711864,0.677966,0.483471
43,433,41212,92,47,18,27,0.510870,0.195652,0.293478,1.728261,0.543478,0.498866
44,433,3412,47,25,8,14,0.531915,0.170213,0.297872,1.765957,0.574468,0.506742


## Example Strategy Report

This imports the same rule-based function used by the script.

In [6]:
from formation_analysis import add_smoothed_metrics, generate_strategy, summarize_matchups

full_summary = add_smoothed_metrics(summarize_matchups(rows))
print(generate_strategy('433', '442', full_summary))

Strategy report: 433 vs 442

Historical matchup performance in this dataset is descriptive, not causal.
Our formation: 433
Opponent formation: 442
Historical games: 331
Win rate: 0.55
Points per game: 1.88
Goal difference per game: 0.97
Interpretation: favorable

Simple tactical suggestions:
- Use the three-player midfield to create central overloads.
- Be careful against two-striker pressure on center backs.
- Wingers should pin the opponent fullbacks.
- The defensive midfielder must protect second balls and counterattacks.
